<a href="https://colab.research.google.com/github/psaesha/llm-evaluability-measurement/blob/main/model_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# !uv venv
!source .venv/bin/activate
!uv pip install -U vllm --torch-backend=auto

Using Python 3.12.13 environment at: /usr
Resolved 189 packages in 2.79s
Checked 189 packages in 1ms


In [ ]:
from huggingface_hub import login
import os
from google.colab import userdata
import wandb

hf_token = userdata.get("HF_TOKEN")
wandb_api = userdata.get("WANDB_API")

login(hf_token)  # paste your HF token

wandb.login(key=wandb_api)  # paste your W&B token

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: saesha-parekh (saesha-parekhcivicdatalab) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
import time
import requests  # For checking server status
import subprocess

# 1. Kill any existing vLLM processes to ensure a clean start
print("Killing any existing vLLM server processes...")
# Use 'pkill -f' to find and terminate processes by command line
subprocess.run(["pkill", "-f", "vllm serve"], check=False)
time.sleep(2)  # Give a moment for processes to terminate

# 2. Disable FlashInfer JIT sampling (requires nvcc which is not available on this runtime)
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

# Start vLLM server in the background using nohup and &
# Redirect stdout and stderr to a log file for debugging
vllm_log_file = "vllm_server.log"
vllm_command = f"""
nohup vllm serve Qwen/Qwen3.5-9B \
  --tensor-parallel-size 1 \
  --max-model-len 262144 \
  --reasoning-parser qwen3 \
  --language-model-only \
  --gpu-memory-utilization 0.75 \
  --dtype float16 \
  --trust-remote-code \
  --port 8000 \
  > {vllm_log_file} 2>&1 &
"""
print(f"Starting vLLM server (output in {vllm_log_file})... ")
# Execute the command directly in the shell
get_ipython().system(vllm_command)

# 3. Implement a robust waiting mechanism
# Check the log file for startup message or attempt a connection
server_ready = False
max_wait_time = 500  # Maximum wait time in seconds (5 minutes)
start_time = time.time()
print(
    "Waiting for vLLM server to become ready (checking log and port, tailing log file)..."
)

last_log_size = 0
while not server_ready and (time.time() - start_time) < max_wait_time:
    if os.path.exists(vllm_log_file):
        current_log_size = os.path.getsize(vllm_log_file)
        if current_log_size > last_log_size:
            # Read only new content from the log file
            with open(vllm_log_file, "r") as f:
                f.seek(last_log_size)  # Move to the last read position
                new_content = f.read()
                print(new_content, end="")  # Print new content without extra newline
            last_log_size = current_log_size

        with open(vllm_log_file, "r") as f:
            log_content = f.read()
            if "Application startup complete" in log_content:
                print("Found 'Application startup complete' in log file.")
                server_ready = True  # Tentatively mark as ready from logs

    # Also try to hit a known API endpoint to confirm readiness
    try:
        # Check /v1/models endpoint, which typically lists available models
        response = requests.get("http://localhost:8000/v1/models", timeout=5)
        if response.status_code == 200:
            print("Successfully connected to vLLM server's /v1/models endpoint.")
            server_ready = True
            break  # Exit loop if connection successful
    except requests.exceptions.ConnectionError:
        pass  # Server not yet ready to accept connections
    except requests.exceptions.Timeout:
        pass  # Connection timed out, still not ready
    except Exception as e:
        # Catch other potential request errors
        print(f"Warning: Error checking vLLM server status: {e}")

    if not server_ready:
        time.sleep(5)  # Wait 5 seconds before checking again

if server_ready:
    print("✅ vLLM server is now ready.")
    # Print any remaining new logs before confirming
    if os.path.exists(vllm_log_file):
        current_log_size = os.path.getsize(vllm_log_file)
        if current_log_size > last_log_size:
            with open(vllm_log_file, "r") as f:
                f.seek(last_log_size)
                print(f.read(), end="")
    print("---------------------------------------")
else:
    print("❌ vLLM server did not become ready within the allotted time.")
    if os.path.exists(vllm_log_file):
        with open(vllm_log_file, "r") as f:
            print("--- Full vLLM Server Log ---")
            print(f.read())
            print("----------------------------")
    # If the server didn't start, we might want to kill it again just in case
    subprocess.run(["pkill", "-f", "vllm serve"], check=False)

Killing any existing vLLM server processes...
Starting vLLM server (output in vllm_server.log)... 
Waiting for vLLM server to become ready (checking log and port, tailing log file)...
(APIServer pid=4871) INFO 06-30 21:28:13 [api_utils.py:339] 
(APIServer pid=4871) INFO 06-30 21:28:13 [api_utils.py:339]        █     █     █▄   ▄█
(APIServer pid=4871) INFO 06-30 21:28:13 [api_utils.py:339]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.24.0
(APIServer pid=4871) INFO 06-30 21:28:13 [api_utils.py:339]   █▄█▀ █     █     █     █  model   Qwen/Qwen3.5-9B
(APIServer pid=4871) INFO 06-30 21:28:13 [api_utils.py:339]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=4871) INFO 06-30 21:28:13 [api_utils.py:339] 
(APIServer pid=4871) INFO 06-30 21:28:13 [api_utils.py:273] non-default args: {'model_tag': 'Qwen/Qwen3.5-9B', 'model': 'Qwen/Qwen3.5-9B', 'trust_remote_code': True, 'dtype': 'float16', 'max_model_len': 262144, 'reasoning_parser': 'qwen3', 'gpu_memory_utilization': 0.75, 'language_model_only': True}
(API

In [19]:
from openai import OpenAI

# Configured by environment variables
client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")

messages = [
    {"role": "user", "content": "hello"},
]

chat_response = client.chat.completions.create(
    model="Qwen/Qwen3.5-9B",
    messages=messages,
    max_tokens=81920,
    temperature=1.0,
    n=5,
    logprobs=True,
    top_logprobs=5,
    reasoning_effort="none",
    top_p=0.95,
    presence_penalty=1.5,
    extra_body={
        "top_k": 20,
    },
)
print("Chat response:", chat_response)

Chat response: ChatCompletion(id='chatcmpl-b32b2b9fa8143cc8', choices=[Choice(finish_reason='stop', index=0, logprobs=ChoiceLogprobs(content=[ChatCompletionTokenLogprob(token='Hello', bytes=[72, 101, 108, 108, 111], logprob=-0.0006040894077159464, top_logprobs=[TopLogprob(token='Hello', bytes=[72, 101, 108, 108, 111], logprob=-0.0006040894077159464), TopLogprob(token='Hi', bytes=[72, 105], logprob=-8.391228675842285), TopLogprob(token='hello', bytes=[104, 101, 108, 108, 111], logprob=-8.672478675842285), TopLogprob(token='你好', bytes=[228, 189, 160, 229, 165, 189], logprob=-10.109978675842285), TopLogprob(token='>Hello', bytes=[62, 72, 101, 108, 108, 111], logprob=-10.125603675842285)]), ChatCompletionTokenLogprob(token='!', bytes=[33], logprob=-0.0003301552205812186, top_logprobs=[TopLogprob(token='!', bytes=[33], logprob=-0.0003301552205812186), TopLogprob(token=' there', bytes=[32, 116, 104, 101, 114, 101], logprob=-9.117517471313477), TopLogprob(token=',', bytes=[44], logprob=-9.195

In [20]:
chat_response.__dict__

{'id': 'chatcmpl-b32b2b9fa8143cc8',
 'choices': [Choice(finish_reason='stop', index=0, logprobs=ChoiceLogprobs(content=[ChatCompletionTokenLogprob(token='Hello', bytes=[72, 101, 108, 108, 111], logprob=-0.0006040894077159464, top_logprobs=[TopLogprob(token='Hello', bytes=[72, 101, 108, 108, 111], logprob=-0.0006040894077159464), TopLogprob(token='Hi', bytes=[72, 105], logprob=-8.391228675842285), TopLogprob(token='hello', bytes=[104, 101, 108, 108, 111], logprob=-8.672478675842285), TopLogprob(token='你好', bytes=[228, 189, 160, 229, 165, 189], logprob=-10.109978675842285), TopLogprob(token='>Hello', bytes=[62, 72, 101, 108, 108, 111], logprob=-10.125603675842285)]), ChatCompletionTokenLogprob(token='!', bytes=[33], logprob=-0.0003301552205812186, top_logprobs=[TopLogprob(token='!', bytes=[33], logprob=-0.0003301552205812186), TopLogprob(token=' there', bytes=[32, 116, 104, 101, 114, 101], logprob=-9.117517471313477), TopLogprob(token=',', bytes=[44], logprob=-9.195642471313477), TopLogp